imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

load data set

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income"
]

df = pd.read_csv(
    "adult.data",
    header=None,
    names=columns,
    na_values="?",
    skipinitialspace=True
)

df = df.dropna()
df["income_binary"] = (df["income"].str.strip() == ">50K").astype(int)

df.head()


Train/Test Split + Column Types

In [ ]:
X = df.drop(columns=["income", "income_binary"])
y = df["income_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline([("scaler", StandardScaler())])
categorical_transformer = Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Basic EDA

In [ ]:
print("Rows, Columns:", df.shape)
print(df["income"].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="income")
plt.title("Income Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(df["age"], kde=True)
plt.title("Age")

plt.subplot(1,2,2)
sns.histplot(df["hours_per_week"], kde=True)
plt.title("Hours per Week")
plt.tight_layout()
plt.show()


Evaluation Function

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test, name):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("="*50)
    print(name)
    print("train acc :", accuracy_score(y_train, y_train_pred))
    print("test acc  :", accuracy_score(y_test, y_test_pred))
    print("precision :", precision_score(y_test, y_test_pred))
    print("recall    :", recall_score(y_test, y_test_pred))
    print("f1        :", f1_score(y_test, y_test_pred))
    print("roc_auc   :", roc_auc_score(y_test, y_proba))
    print("\nreport:")
    print(classification_report(y_test, y_test_pred, digits=4))

    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(4,4))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(f"{name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title(f"{name} ROC Curve")
    plt.tight_layout()
    plt.show()


Logistic Regression

In [ ]:
log_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=-1))
])

log_pipe.fit(X_train, y_train)

evaluate(log_pipe, X_train, y_train, X_test, y_test, "Logistic Regression")


Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_pipe.fit(X_train, y_train)

evaluate(rf_pipe, X_train, y_train, X_test, y_test, "Random Forest")


Feature Names

In [ ]:
def get_feature_names(preprocessor):
    nums = preprocessor.transformers_[0][2]
    cat_pipe = preprocessor.transformers_[1][1]
    onehot = cat_pipe.named_steps["onehot"]
    cats = onehot.get_feature_names_out(preprocessor.transformers_[1][2])
    return np.concatenate([nums, cats])

feature_names = get_feature_names(log_pipe.named_steps["prep"])


Logistic Regression Coefficients

In [ ]:
log_model = log_pipe.named_steps["clf"]
coefs = log_model.coef_[0]

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs", ascending=False).head(20)

plt.figure(figsize=(8,6))
sns.barplot(data=coef_df, x="abs", y="feature")
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

coef_df


Random Forest Importances

In [ ]:
rf_model = rf_pipe.named_steps["clf"]
imp = rf_model.feature_importances_

imp_df = pd.DataFrame({"feature": feature_names, "importance": imp})
imp_df = imp_df.sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(8,6))
sns.barplot(data=imp_df, x="importance", y="feature")
plt.title("Top Random Forest Feature Importances")
plt.tight_layout()
plt.show()

imp_df
